# 기억력 스포츠 선수 성적 분석 — 1일차 실행용

**기존 졸업작품 주제를 유지한 최소 구현: 종목별 데이터 → 정규화 → 상관관계 → K-means → 시각화·훈련 검토**

이 노트북은 우선 **공식 사이트에서 수작업으로 검증한 2025년 대회 14명 발췌 자료**로 실행됩니다. **전체 선수를 대표하는 표본이 아니므로** 여기서 나온 상관계수·군집은 연구 결론으로 사용하지 않습니다. 프로그램 작동 증빙을 위한 연습 자료입니다.

공식 출처: [무작위 단어](https://www.worldmemorychampionships.com/competitions-2025-random-words/) / [스피드 넘버](https://www.worldmemorychampionships.com/competitions-2025-speed-numbers/) / [추상 이미지](https://www.worldmemorychampionships.com/competitions-2025-abstract-images/) / [스피드 카드](https://www.worldmemorychampionships.com/competitions-2025-speed-cards/)

성적은 모두 **Points(공식 환산점수)**입니다. Speed Cards는 `Points (highest)`입니다. 개인 이름 대신 공개된 선수 ID만 보관합니다.

전체 공식 성적을 적법한 방법으로 확보한 뒤 아래에서 설명한 CSV로 교체할 수 있습니다.

In [ ]:
# 1. 라이브러리 설치 (Colab에서 실행)
!pip -q install pandas numpy scikit-learn matplotlib gradio

In [ ]:
# 2. Google Drive 연결. PC를 다시 켜도 파일을 되찾을 수 있게 저장합니다.
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/MemorySports_Graduation')
PROJECT.mkdir(parents=True, exist_ok=True)
print('작업 폴더:', PROJECT)

## 3. 분석 데이터 준비

기본은 공식 결과에서 발췌한 14명 기록입니다.

이후 분석용 전체 자료를 확보하면 Drive의 `MemorySports_Graduation/full_scores.csv`에 저장하세요. 열 이름은 `ID, Category, Random Words, Speed Numbers, Abstract Images, Speed Cards`이어야 합니다. 그 파일이 있으면 **자동으로 전체 자료를 사용**합니다.

주의: 전체 분석에는 가능한 모든 대회 참가자 기록이 포함되어야 합니다. 상위권 선수만 임의 추출하면 선택 편향이 생깁니다.

In [ ]:
import pandas as pd
import numpy as np

EXCERPT = [
("J030","Junior",815,886,902,662),
("J041","Junior",678,756,493,637),
("J037","Junior",493,714,474,636),
("J023","Junior",546,659,697,674),
("A048","Adult",872,682,426,601),
("A055","Adult",639,549,778,408),
("J036","Junior",746,584,437,456),
("A040","Adult",239,390,841,265),
("J040","Junior",397,552,480,514),
("J045","Junior",281,529,236,476),
("K053","Kid",490,490,293,431),
("A057","Adult",233,188,93,229),
("K051","Kid",212,403,326,495),
("A052","Adult",439,438,635,747),
]
EVENTS = ["Random Words", "Speed Numbers", "Abstract Images", "Speed Cards"]
COLUMNS = ["ID", "Category"] + EVENTS
excerpt_file = PROJECT / 'official_excerpt_14.csv'
sample_df = pd.DataFrame(EXCERPT, columns=COLUMNS)
sample_df.to_csv(excerpt_file, index=False, encoding='utf-8-sig')

full_file = PROJECT / 'full_scores.csv'
if full_file.exists():
    df = pd.read_csv(full_file, dtype={'ID':str})
    SOURCE = '전체 자료 CSV (별도 확보)'
else:
    df = sample_df.copy()
    SOURCE = '공식 기록 14명 비대표 발췌 (시연 전용)'
missing = set(COLUMNS) - set(df.columns)
if missing:
    raise ValueError(f'CSV 열이 부족합니다: {sorted(missing)}')

df["ID"] = df["ID"].astype(str).str.strip()
for e in EVENTS:
    df[e] = pd.to_numeric(df[e], errors='coerce')
df = df.drop_duplicates(subset=["ID"], keep=False).copy()
if df.empty:
    raise ValueError('중복 제거 후 분석 가능한 데이터가 없습니다.')
print('자료 종류:', SOURCE)
print('분석 행 수:', len(df))
print('종목별 결측:', df[EVENTS].isna().sum().to_dict())
display(df.head(10))

## 4. 종목별 점수 비교와 상관관계

종목별 환산 점수 자체를 그대로 비교하지 않고, **같은 대회·같은 종목의 참가자 안에서 백분위 순위**를 계산합니다. 숫자가 클수록 해당 대회 참가자 사이에서 상대 성적이 높습니다.

그다음 스피어만 상관계수로 종목별 상대 순위 관계를 탐색합니다.

In [ ]:
import matplotlib.pyplot as plt

complete = df.dropna(subset=EVENTS).copy()
print('4종목 모두 점수 있는 선수:', len(complete), '/', len(df))
if len(complete) < 6:
    raise ValueError('4종목 점수가 모두 있는 선수가 6명 미만입니다. 자료를 추가해 주세요.')

ranks = complete[EVENTS].rank(pct=True, method='average') * 100
ranks.index = complete['ID'].values
corr = ranks.corr(method='spearman')
print('종목별 순위 상관계수 (표본의 성적 관계이며 인과관계가 아님):')
display(corr.round(3))
corr.to_csv(PROJECT/'event_correlations.csv', encoding='utf-8-sig')

fig, ax = plt.subplots(figsize=(6.5,4.5))
im = ax.imshow(corr.to_numpy(), vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(EVENTS)), EVENTS, rotation=30, ha='right')
ax.set_yticks(range(len(EVENTS)), EVENTS)
for i in range(len(EVENTS)):
    for j in range(len(EVENTS)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center')
fig.colorbar(im, ax=ax, label='Spearman correlation')
fig.tight_layout()
fig.savefig(PROJECT/'01_correlation.png', dpi=150)
plt.show()

## 5. K-means로 상대적인 성적 패턴의 유형 탐색

순위 백분위를 계산한 뒤 각 선수의 평균 순위를 빼서 **상대적으로 강한 종목의 패턴**을 추출합니다.

K=2,3,4 등 여러 값을 시도하고 **실루엣 계수**를 비교합니다. 군집 개수와 이름을 연구자가 무조건 가정하지 않습니다. 실루엣 값이 낮으면 뚜렷한 유형이 없다고 판단할 수도 있습니다.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X = ranks.to_numpy(dtype=float)
X_pattern = X - X.mean(axis=1, keepdims=True)
scores = {}
for k in range(2, min(6, len(complete)-1)):
    labels = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(X_pattern)
    if 1 < len(set(labels)) < len(complete):
        scores[k] = silhouette_score(X_pattern, labels)
if not scores:
    raise ValueError('군집화 평가가 어려운 데이터입니다.')

best_k = max(scores, key=scores.get)
model = KMeans(n_clusters=best_k, n_init=20, random_state=42)
cluster = model.fit_predict(X_pattern)
result = complete[['ID','Category']].copy()
result['Cluster'] = cluster
for e in EVENTS:
    result[e] = ranks[e].to_numpy()

print('K별 실루엣 계수:', {k:round(v,3) for k,v in scores.items()})
print('최고 점수의 K:', best_k, '(값이 낮으면 구분이 희미할 수 있음)')
print('그룹별 평균 종목 백분위:')
display(result.groupby('Cluster')[EVENTS].mean().round(1))
result.to_csv(PROJECT/'02_cluster_results.csv', index=False, encoding='utf-8-sig')

fig,ax=plt.subplots(figsize=(5.5,3))
ax.bar([str(k) for k in scores], list(scores.values()))
ax.set_xlabel('Number of clusters (K)')
ax.set_ylabel('Silhouette score')
ax.set_title('K-means cluster evaluation')
fig.tight_layout()
fig.savefig(PROJECT/'02_silhouette.png',dpi=150)
plt.show()

## 6. 개별 선수의 결과 그래프

선수 ID를 바꾸면 종목별 상대 성적과 **훈련을 검토할 종목**을 보여줍니다. 가장 낮은 종목을 제시하는 것은 규칙 기반의 제안이며 **훈련 효과가 검증된 추천이 아닙니다.**

In [ ]:
import gradio as gr

def player_report(player_id):
    row = result.loc[result['ID'] == player_id]
    if row.empty:
        return None, "선수 ID를 선택하세요."
    row = row.iloc[0]
    vals = row[EVENTS].astype(float)
    fig,ax=plt.subplots(figsize=(6,3.5))
    ax.bar(EVENTS, vals.values)
    ax.set_ylim(0,100)
    ax.set_ylabel('Percentile within loaded athletes')
    ax.set_xticks(range(len(EVENTS)), EVENTS, rotation=20, ha='right')
    ax.set_title(f'ID {player_id} | Cluster {int(row.Cluster)}')
    fig.tight_layout()
    weak = vals.idxmin()
    note = f"**상대적으로 낮은 종목: {weak}**\n\n"
    note += f"- 현재 데이터 내부 순위 백분위: **{vals.min():.1f}**\n"
    note += "- 해당 종목을 기초 훈련으로 검토할 수 있습니다. 개인별 최적 훈련이라는 뜻은 아닙니다.\n"
    note += f"- **데이터 제한:** {SOURCE}. 소규모 발췌 자료라면 연구적 결론을 내리지 마세요."
    return fig,note

with gr.Blocks() as demo:
    gr.Markdown('# 기억력 스포츠 종목별 성적 분석 — 초기 프로토타입')
    gr.Markdown(f'**현재 자료:** {SOURCE}')
    choose = gr.Dropdown(choices=result['ID'].tolist(), value=result['ID'].iloc[0], label='선수 ID')
    plot = gr.Plot(label='선수 성적')
    text = gr.Markdown()
    gr.Button('분석하기').click(player_report, inputs=choose, outputs=[plot,text])
demo.launch(share=False)

## 7. 오늘의 실행 결과 기록

완료 체크:
- [ ] Drive 연결 및 `official_excerpt_14.csv` 저장 확인
- [ ] `01_correlation.png` 생성 확인
- [ ] `02_silhouette.png` 생성 확인
- [ ] Gradio 화면에서 선수 ID 선택·분석 실행
- [ ] 실제 전체 경기 결과 수집 방법/허용 범위 확인
- [ ] Github에 오늘의 코드·오류·결과 기록

**보고서 문장 예시:** "기존 계획의 종목별 분석 프로토타입을 구현하고, 시연 단계에서는 공개된 공식 기록의 일부를 발췌해 처리 파이프라인을 검증하였다. 발췌 표본의 군집 분석 결과는 대표성 부족으로 일반화하지 않았다."

**다음 단계:** 공식 전체 참가자의 성적을 적법한 방법으로 확보하여 `full_scores.csv`로 저장한 뒤 동일 노트북에서 실제 분석 수행. 데이터 허용 범위, 결측 처리, 표본 크기, 선택 편향을 함께 보고.